[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_6_Mars_Rover_Capstone_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 🚀 اللاب 6: المشروع النهائي: تقطيع الشبكة والملاحة الذاتية على المريخ
### *غرفة التحكم: رفع كود الملاحة لكنترول مركبة الفضاء*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الربط النهائي الشامل (من النظرية الرياضية إلى كمبيوتر ملاحة المريخ)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $\nabla^2\phi = 0$ | "معادلة لابل المستمرة التي أثبتنا جذورها $m=\pm i$ في اللاب 4." | `(D_x^2 + D_y^2)phi = 0` |
| $\phi_{i,j} = \frac{1}{4}(\phi_{\text{up}} + \phi_{\text{down}} + \dots)$ | "صيغة الـ 5-Point Stencil: كل نقطة هي متوسط جيرانها الأربعة." | `avg = (roll_up + roll_down + ...) / 4` |
| $\frac{\partial\phi}{\partial n} = 0$ | "جدران عازلة (Neumann): الماء يلتف حول الحفر دون اختراقها." | `phi[~free] = 0; cnt = active_neighbors` |
| $\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/N)}$ | "معامل الاسترخاء الفائق (SOR) لتسريع الحل 66 ضعفاً." | `phi = (1 - w)*old + w*avg` |

---
## 1. المعنى البسيط: "الترقية الكبرى: من الورقة إلى 25,600 نقطة"
في اللابات السابقة، حللنا المعادلات بالورقة والقلم على ورقة بيضاء ملساء.
لكن على سطح المريخ، صورة التضاريس الرقمية المأخوذة من قمر ناسا الصناعي (HiRISE DEM) تحتوي على **25,600 نقطة ارتفاع، وأكثر من 3,400 حفرة وصخرة غير منتظمة الشكل**!
* يستحيل على أي عالم في العالم كتابة معادلة ورقية واحدة تغطي 10,000 جدار حفرة متعرج.
* **الحل الهندسي:** نحول المشتقات المتصلة إلى شبكة نقاط متقطعة (Discretization) ونترك بايثون يحل الـ 25,600 معادلة في ثوانٍ معدودة!

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** لأن مركبة الفضاء لا تمتلك فريق إنقاذ بشري على المريخ. إذا علقت داخل حفرة، تنتهي المهمة وتضيع ملايين الدولارات.
* **ماذا تفيد مركبة المريخ؟** بفضل نظرية **The Maximum Principle** لمعادلة لابل، يضمن كمبيوتر الملاحة رياضياً بنسبة 100% أن المركبة لن تعلق في أي فخ محلي وستصل إلى الهدف بسلام.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab6

%matplotlib inline
print("تم تهيئة اللاب 6 النهائي بنجاح! جاهز لتشغيل تسلسل إطلاق المهمة.")


---
## 3. الوصفة الرياضية: استنتاج الـ 5-Point Stencil من متسلسلة تايلور

لنقرب المشتقة الثانية $\frac{\partial^2\phi}{\partial x^2}$ عند النقطة $(i, j)$ باستخدام متسلسلة تايلور للأمام والخلف:
$$\phi_{i+1, j} = \phi_{i,j} + h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} + \mathcal{O}(h^3)$$
$$\phi_{i-1, j} = \phi_{i,j} - h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} - \mathcal{O}(h^3)$$

بجمع المعادلتين معاً، تحذف المشتقات الفردية، وبإعادة الترتيب:
$$\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} = \frac{\phi_{i+1, j} - 2\phi_{i,j} + \phi_{i-1, j}}{h^2}$$

وبنفس الطريقة في اتجاه $y$:
$$\left.\frac{\partial^2\phi}{\partial y^2}\right|_{i,j} = \frac{\phi_{i, j+1} - 2\phi_{i,j} + \phi_{i, j-1}}{h^2}$$

بالتعويض في معادلة لابل $\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ والضرب في $h^2$:
$$(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}) - 4\phi_{i,j} = 0$$
$$\implies \phi_{i,j} = \frac{1}{4}\left(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}\right)$$
> *"كل خلية في الخريطة هي ببساطة المتوسط الحسابي لجيرانها الأربعة المباشرين!"*


---
## 4. 📝 تحدي الورقة والقلم 1 (الحساب اليدوي لشبكة 4 نقاط)

> 💡 *(إذا واجهتك صعوبة، راجع `INSTRUCTOR_ANSWER_KEY.md` القسم 6.1).*

لدينا شبكة $4 \times 4$ بشروط حدية:
* الحد العلوي (البداية): $\phi = 1.0$
* الحد السفلي (الهدف): $\phi = 0.0$
* الحد الأيمن والأيسر (جدران معزولة): $\phi = 0.5$
* الخلايا الداخلية الأربعة $A, B, C, D$ قيمتها الابتدائية $0.0$.

احسب التحديث الأول يدوياً (Sweep 1) للخلية $A(1, 1)$ والخلية $B(1, 2)$:
$$\phi_A^{(1)} = \frac{1.0 + 0.0 + 0.5 + 0.0}{4} = ?$$

أدخل القيمتين العشريتين المحسوبتين باليد أدناه:


In [ ]:
# خلية تقييم: حساب الشبكة اليدوي
### START CODE HERE ### (≈ 2 lines of code)
# أدخل القيمتين العشريتين المحسوبتين باليد (مثال: 0.375)
phi_A = None
phi_B = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_1(phi_A, phi_B)


---
## 5. 🚀 تحدي جسر مركبة المريخ: الحساب النظري لمعامل SOR الأمثل

على الورق استغرق حساب 4 خلايا دقيقتين. على شبكة $N = 16$، ما هي القيمة النظرية لمعامل الاسترخاء الأمثل $\omega_{\text{opt}}$ وفقاً لنظرية يونغ (Young's Theorem)؟
$$\omega_{\text{opt}} = \frac{2}{1 + \sin\left(\frac{\pi}{N}\right)}$$

احسب قيمة $\omega_{\text{opt}}$ لـ $N = 16$ على ورقتك وأدخلها أدناه:


In [ ]:
# خلية تقييم: معامل أوميجا الأمثل لـ N=16
### START CODE HERE ### (≈ 1 line of code)
# احسب قيمة أوميجا المثلى لـ N=16 (صيغة: 2.0 / (1.0 + np.sin(np.pi / 16)))
omega_16 = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_2(omega_16)


---
## 6. 💻 التطبيق البرمجي ببايثون: خوارزمية Red-Black SOR مع جدران نيومان العازلة

سنكتب الآن خوارزمية الاسترخاء الفائق المسرعة بنظام رقعة الشطرنج (Red-Black Ordering) مع تطبيق شروط نيومان العازلة ($\frac{\partial\phi}{\partial n} = 0$) على جدران حفر المريخ:


In [ ]:
# دالة تقييم برمجية: sor_neumann_step
def sor_neumann_step(phi, fixed, free, omega=1.5, tol=1e-5, maxit=5000):
    # خوارزمية Red-Black SOR مع جدران نيومان العازلة للحفر
    phi = phi.copy()
    n, m = phi.shape
    f = free.astype(float)
    
    cnt = sum(np.roll(f, s, a) for a in (0, 1) for s in (1, -1))
    cnt = np.maximum(cnt, 1.0)
    
    yy, xx = np.mgrid[0:n, 0:m]
    parity = (xx + yy) % 2
    masks = [(parity == c) & free & ~fixed for c in (0, 1)]
    phi[~free] = 0.0
    hist = []
    
    ### START CODE HERE ### (≈ 8-12 lines of code)
    # حلقة تكرارية بعدد maxit:
    # 1. d = 0.0
    # 2. لكل قناع mk في masks:
    #        احسب المتوسط الخماسي: avg = (np.roll(pf, 1, 0) + np.roll(pf, -1, 0) + np.roll(pf, 1, 1) + np.roll(pf, -1, 1)) / cnt
    #        حدث الجهد: phi[mk] = (1 - omega)*old + omega*avg[mk]
    #        احسب أكبر تغيير: d = max(d, float(np.abs(phi[mk] - old).max()))
    # 3. hist.append(d)
    # 4. إذا كان d < tol: اخرج من الحلقة
    pass
    ### END CODE HERE ###
    
    return phi, hist

# تشغيل الاختبار الآلي:
test_lab6.test_sor_neumann_step(sor_neumann_step)


---
## 7. 🌉 الجسر الرياضي والفيزيائي: لماذا نستبدل معادلات الروبوتات القديمة بـ PDE؟

قبل تشغيل المحاكاة الكبرى على **25,600 نقطة من تضاريس المريخ**، دعنا نفهم الرابط الجوهري بين ما كتبته بيدك على الورق وفيزياء ملاحة الفضاء:

### 1. ما هي المعادلة التي تبدأ بها الروبوتات التقليدية؟ (مجال الجهد الاصطناعي - APF)
في الروبوتات العادية وألعاب الفيديو، يبدأ المبرمجون بصيغة جبرية بديهية:
$$U(\mathbf{x}) = U_{\text{att}}(\mathbf{x}) + U_{\text{rep}}(\mathbf{x})$$
* **بئر جذب الهدف ($U_{\text{att}}$):** الهدف يجذب المركبة كأنه نابض جاذبية: $\mathbf{F}_{\text{att}} = -\nabla U_{\text{att}} = -k_{\text{att}}(\mathbf{x} - \mathbf{x}_{\text{goal}})$.
* **جدار تنافر الحفر ($U_{\text{rep}}$):** الحفر تدفع المركبة بعيداً كأنها شحنة كهربائية متنافرة: $\mathbf{F}_{\text{rep}} = -\nabla U_{\text{rep}}$.

وتتحرك المركبة كأنها كرة تتدحرج تحت تأثير محصلة القوى: $\mathbf{F}_{\text{net}} = \mathbf{F}_{\text{att}} + \mathbf{F}_{\text{rep}}$.

---

### 2. لماذا تفشل هذه المعادلة فيزيائياً؟ (فخ الحفر المسدودة 🪤)
تخيل أن مركبة المريخ دخلت حفرة على شكل حرف **U** وكان موقع الهدف العلمي يقع خلف الجدار الخلفي للحفرة مباشرة:
* مع اقتراب المركبة من الجدار نحو الهدف، تزداد قوة الجذب للأمام ($\mathbf{F}_{\text{att}}$).
* وفي نفس الوقت، تزداد قوة التنافر من الجدار للخلف ($\mathbf{F}_{\text{rep}}$).
* في نقطة ما داخل الحفرة، **تتساوى القوتان تماماً في المقدار وتتعاكسان في الاتجاه**:
  $$\mathbf{F}_{\text{att}} + \mathbf{F}_{\text{rep}} = \mathbf{0} \implies \nabla U = \mathbf{0}$$

> ⚠️ **الكارثة الهندسية:** تتلاشى القوة تماماً وتصبح صفراً! تتشكل حفرة جهد صناعية (Local Minimum). تعلق المركبة وتتجمد مكانها حتى تنفد بطاريتها وتفشل المهمة الفضائية بالكامل!

---

### 3. لماذا نحتاج لتغيير المعادلة واستخدام معادلة تفاضلية جزئية (PDE)؟
لأن الجمع الجبري البسيط $U_{\text{att}} + U_{\text{rep}}$ يسمح بتكون قيعان ومنخفضات عشوائية في الفراغ.
ولمنع الفخاخ للأبد، نبحث عن ظاهرة طبيعية **يستحيل فيزيائياً أن تعلق في حفرة**: **إنها فيزياء تدفق المياه 🌊**!
الماء يملأ الحفرة، ويرتفع ضغطه ضد الجدران، وينساب بسلاسة حول العائق نحو المصب دون أن يتجمد في بركة راكدة بالمنتصف.

---

### 4. ما هي المعادلة البديلة وما دورها في المشروع؟
المعادلة الفيزيائية التي تحكم ضغط الموائع والجهد الكهربائي الساكن هي **معادلة لابل (Laplace's Equation)**:
$$\nabla^2 \phi = \frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0$$

#### دورها في مشروع المريخ:
1. **النطاق ($\Omega$):** نحول خريطة المريخ إلى حقل جهد $\phi(x, y)$.
2. **المصدر والمصب (Dirichlet BCs):** البداية عند أعلى جهد $\phi = 1.0$، والهدف عند أدنى جهد $\phi = 0.0$.
3. **جدران الحفر (Neumann BCs):** جدران معزولة تماماً لا يخترقها الماء $\frac{\partial\phi}{\partial n} = 0$.
4. **الضمان الرياضي القاطع (The Strong Maximum Principle من اللاب 4):**
   الدالة التوافقية **يستحيل أن تمتلك قيعاناً صغرى محلية داخل النطاق**! القمم والقيعان تتواجد فقط على البداية والهدف.
   وبالتالي، تيار الانحدار $-\nabla\phi$ **لا ينعدم أبداً** في المساحات الحرة.
5. **مسار حركة العجلات (Lagrange's Streamline Method من اللاب 2):**
   تنساب عجلات المركبة مع خطوط التيار $\frac{dx}{u} = \frac{dy}{v}$ لتصل إلى الهدف بضمان رياضي 100%!

---
## 8. 🎨 الإطلاق النهائي للمهمة: الملاحة الذاتية على تضاريس المريخ

الآن بعد أن أتقنا الرياضيات والفيزياء، شغّل المحاكاة وشاهد كيف توجه خوارزمية لابل المركبة بنجاح حول الحفر على **25,600 نقطة تضاريس حقيقية من وكالة ناسا**:


In [ ]:
from rover_path_finder.terrain import make_terrain
from rover_path_finder.planners import neumann_potential, descend, apf_potential, trap_map

# 1. توليد تضاريس مريخية 100x100
terrain_data = make_terrain(N=100, seed=3, rover_radius=2.0)
terrain = terrain_data['h']
hazards = terrain_data['hazard']
start = terrain_data['start']
goal = terrain_data['goal']

# 2. حل معادلة لابل
phi, sweeps, elapsed = neumann_potential(hazards, start=start, goal=goal)

# 3. تتبع مسار الانحدار مع خطوط الانسياب
path, reached = descend(phi, hazards, start=start, goal=goal)

# 4. رسم حقل الجهد والمسار
fig, ax = plt.subplots(figsize=(10, 8))
cs = ax.contourf(phi, levels=30, cmap='viridis')
plt.colorbar(cs, label="Harmonic Potential $\phi(x, y)$")
ax.contour(hazards, levels=[0.5], colors='red', linewidths=1.5)

px = [p[1] for p in path]
py = [p[0] for p in path]
ax.plot(px, py, color='cyan', linewidth=3, label="Rover Trajectory (Laplace Streamline)")
ax.plot(start[1], start[0], 'go', markersize=10, label="Start ($\phi=1$)")
ax.plot(goal[1], goal[0], 'r*', markersize=14, label="Goal ($\phi=0$)")

ax.set_title("Ares Rover Navigation: Laplace Streamlines Around Martian Craters", fontsize=14)
ax.legend(loc='upper left')
plt.show()

print(f"حالة المهمة: هل وصلت المركبة للهدف بنجاح؟ {reached} في {len(path)} خطوة ({elapsed:.2f} ثانية)!")


---
## 9. 🪤 الاختبار الحاسم: مقارنة فخ الحفر المسدودة (U-Shaped Trap Test)

شغّل الخلية التالية لمشاهدة المقارنة المباشرة بين الروبوت التقليدي (APF) وهو يعلق داخل الحفرة، وبين معادلة لابل التوافقية وهي تلتف بنجاح وتخرج المركبة للأمان:


In [ ]:
u_trap, trap_start, trap_goal = trap_map(N=100)

# APF
apf_field = apf_potential(u_trap, goal=trap_goal)
apf_path, apf_reached = descend(apf_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=200)

# Laplace
laplace_field, _, _ = neumann_potential(u_trap, start=trap_start, goal=trap_goal)
laplace_path, laplace_reached = descend(laplace_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=400)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# رسم الروبوت الكلاسيكي
axes[0].contourf(apf_field, levels=25, cmap='magma')
axes[0].plot([p[1] for p in apf_path], [p[0] for p in apf_path], 'r-', linewidth=3, label="APF Path")
axes[0].plot(trap_start[1], trap_start[0], 'go', markersize=8)
axes[0].plot(trap_goal[1], trap_goal[0], 'y*', markersize=12)
axes[0].set_title("Classical Robotics (APF): TRAPPED in Local Minimum! ❌", fontsize=12)
axes[0].legend()

# رسم معادلة لابل
axes[1].contourf(laplace_field, levels=25, cmap='viridis')
axes[1].plot([p[1] for p in laplace_path], [p[0] for p in laplace_path], 'c-', linewidth=3, label="Laplace Path")
axes[1].plot(trap_start[1], trap_start[0], 'go', markersize=8, label="Start")
axes[1].plot(trap_goal[1], trap_goal[0], 'y*', markersize=12, label="Goal")
axes[1].set_title("Laplace PDE: Maneuvers Cleanly Out of the Trap! ✅", fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

print("نتائج المقارنة الحاسمة:")
print(f"  • الروبوت التقليدي (APF) : هل وصل للهدف؟ {apf_reached} (علق نهائياً داخل الحفرة المسدودة)")
print(f"  • معادلة لابل (Laplace) : هل وصل للهدف؟ {laplace_reached} (خرج بسلام من الحفرة في {len(laplace_path)} خطوة!)")


## 🏆 شهادة التخرج من الكورس

**تهانينا الحارة على إتمام كورس المعادلات التفاضلية الجزئية بنجاح!**

لقد أتقنت الرحلة الهندسية والرياضية الكاملة:
1. **الأسس التحليلية:** كونت المعادلات بحذف الثوابت، وحللت خطوط الانسياب بلاجرانج، وقهرت موجات الرادار بشاربيت، وفهمت أصل معادلة لابل من مؤثرات الرتب العليا.
2. **الترقية الرقمية:** حولت المعادلات إلى شبكة 5-Point Stencil، وأثبت سرعة SOR الفائقة بمقدار 66 ضعفاً.
3. **الملاحة الفضائية على المريخ:** حللت معضلة فخاخ الحفر النيزكية بفضل قانون Maximum Principle، وضمنت ملاحة آمنة وذكية لمركبات الفضاء!
